In [1]:
pip install opencv-python face_recognition numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
import face_recognition
import numpy as np
import os
from datetime import datetime

In [3]:
path = 'Images_dataset'
images = []
classNames = []

for person in os.listdir(path):
    for img_name in os.listdir(f'{path}/{person}'):
        img = cv2.imread(f'{path}/{person}/{img_name}')
        images.append(img)
        classNames.append(person)

In [4]:
import cv2
import numpy as np
import os
import face_recognition
from datetime import datetime

path = 'Images_dataset'
images = []
classNames = []

for person in os.listdir(path):
    for img_name in os.listdir(f'{path}/{person}'):
        img = cv2.imread(f'{path}/{person}/{img_name}')
        images.append(img)
        classNames.append(person)

print("Loaded Names:", set(classNames))

Loaded Names: {'suhan'}


In [5]:
def markAttendance(name):
    with open('attendance.csv', 'a+') as f:
        f.seek(0)
        data = f.readlines()
        names = [line.split(',')[0] for line in data]

        if name not in names:
            now = datetime.now()
            time = now.strftime('%H:%M:%S')
            date = now.strftime('%d/%m/%Y')   # day/month/year
            
            f.writelines(f'\n{name},{date},{time}')

In [6]:
def findEncodings(images):
    encodeList = []
    for img in images:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        encodes = face_recognition.face_encodings(img)
        
        if len(encodes) > 0:
            encodeList.append(encodes[0])
    
    return encodeList

# Generate encodings
encodeListKnown = findEncodings(images)

print("Encoding Complete ✅")

Encoding Complete ✅


In [7]:
cap = cv2.VideoCapture(0)

printed_names = set()   # to avoid repeated printing
unknown_printed = False

while True:
    success, img = cap.read()

    imgSmall = cv2.resize(img, (0,0), None, 0.25, 0.25)
    imgSmall = cv2.cvtColor(imgSmall, cv2.COLOR_BGR2RGB)

    facesCurFrame = face_recognition.face_locations(imgSmall, model='hog')
    encodesCurFrame = face_recognition.face_encodings(imgSmall, facesCurFrame)

    for encodeFace, faceLoc in zip(encodesCurFrame, facesCurFrame):
        matches = face_recognition.compare_faces(encodeListKnown, encodeFace)
        faceDis = face_recognition.face_distance(encodeListKnown, encodeFace)

        matchIndex = np.argmin(faceDis)

        if matches[matchIndex]:
            name = classNames[matchIndex].upper()

            if name not in printed_names:
                printed_names.add(name)

                now = datetime.now()
                time = now.strftime('%H:%M:%S')
                date = now.strftime('%d/%m/%Y')

                print("✅ Detected Person:", name)
                print("🕒 Time:", time)
                print("📅 Date:", date)
                print("📌 Attendance Marked\n")

                markAttendance(name)

        else:
            name = "UNKNOWN"

            if not unknown_printed:
                print("❌ Imposter / Not Detected\n")
                unknown_printed = True

        # Draw box
        y1, x2, y2, x1 = faceLoc
        y1, x2, y2, x1 = y1*4, x2*4, y2*4, x1*4

        color = (0,255,0) if name != "UNKNOWN" else (0,0,255)

        cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
        cv2.putText(img, name, (x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    cv2.imshow('Face Attendance System', img)

    key = cv2.waitKey(1)
    if key == 27 or key & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

✅ Detected Person: SUHAN
🕒 Time: 17:55:50
📅 Date: 08/04/2026
📌 Attendance Marked



In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

uploaded_img = None
output = widgets.Output()  # ← captures all prints/plots

# Upload widget
upload = widgets.FileUpload(accept='image/*', multiple=False)

# Detect button
detect_btn = widgets.Button(description="Detect Face", button_style='success')

display(upload, detect_btn, output)


# 📁 Upload handler
def on_upload_change(change):
    global uploaded_img
    if upload.value:
        with output:
            clear_output()
            try:
                # ✅ NEW API — works in ipywidgets 7.7+ and 8.x
                uploaded_file = upload.value[0]
                file_bytes = np.frombuffer(uploaded_file['content'], np.uint8)
                uploaded_img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
                print("📁 Image uploaded successfully\n")
            except Exception as e:
                print(f"❌ Upload error: {e}")

upload.observe(on_upload_change, names='value')


# 🔍 Detect button logic
def on_detect_click(b):
    global uploaded_img

    with output:
        clear_output()

        if uploaded_img is None:
            print("❌ Please upload an image first\n")
            return

        img = uploaded_img.copy()
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        facesCurFrame = face_recognition.face_locations(imgRGB, model='hog')
        encodesCurFrame = face_recognition.face_encodings(imgRGB, facesCurFrame)

        if len(encodesCurFrame) == 0:
            print("❌ No face detected in the image\n")
            return

        for encodeFace, faceLoc in zip(encodesCurFrame, facesCurFrame):

            faceDis = face_recognition.face_distance(encodeListKnown, encodeFace)

            if len(faceDis) == 0:
                print("❌ No known faces to compare\n")
                continue

            matchIndex = np.argmin(faceDis)
            print(f"🔍 Distance Score: {faceDis[matchIndex]:.4f}  (lower = better match, threshold = 0.5)")

            if faceDis[matchIndex] < 0.5:
                name = classNames[matchIndex].upper()

                now = datetime.now()
                time_str = now.strftime('%H:%M:%S')
                date_str = now.strftime('%d/%m/%Y')

                print("✅ Detected Person:", name)
                print("🕒 Time:", time_str)
                print("📅 Date:", date_str)
                print("📌 Attendance Marked\n")

                markAttendance(name)
                color = (0, 255, 0)

            else:
                name = "UNKNOWN"
                print("❌ Imposter / Not Recognized\n")
                color = (0, 0, 255)

            # Draw rectangle on image
            y1, x2, y2, x1 = faceLoc
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, name, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

        # Show image
        plt.figure(figsize=(8, 6))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title("Face Recognition Result")
        plt.show()


detect_btn.on_click(on_detect_click)

FileUpload(value=(), accept='image/*', description='Upload')

Button(button_style='success', description='Detect Face', style=ButtonStyle())

Output()